# FRL submission — online-appendix exhibits (Tables A1-E1)

Rebuilds the appendix tables of the current FRL manuscript from the artifacts, with asserts against the printed values.
One documented discrepancy: the manuscript's Table A1 upstream counts come from the v2 extraction (flagged for correction).


## Setup


In [ ]:
import json, os
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
ART = "../artifacts"
def A(name):
    with open(os.path.join(ART, name + ".json"), encoding="utf-8") as f: return json.load(f)
def show(df, t): print(t); print("-" * len(t)); print(df.to_string(index=False)); print()
def f4(x): return f"{x:+.4f}"
def ci4(c): return f"[{c[0]:+.4f}, {c[1]:+.4f}]"
def okp(a, b, nd=4): assert round(a, nd) == round(b, nd), (a, b)
print("ready")


ready


## A — Sample flow and stakes (Tables A1-A2)


In [ ]:
u = A("wp13a_universe")["flow"]
show(pd.DataFrame(u), "Sample flow (canonical universe v3)")
st = A("wp10ab")["A_stake"]
rows = [{"Statistic": k, "Value": v} for k, v in (("25th percentile", f"{st['p25']:.4f}"), ("Median", f"{st['median']:.4f}"),
        ("75th percentile", f"{st['p75']:.4f}"), ("90th percentile", f"{st['p90']:.4f}"),
        ("Share >= 30%", f"{st['ge30']:.4f}"), ("Share >= 50%", f"{st['ge50']:.4f}"), ("Observations", st["n_stake"]))]
show(pd.DataFrame(rows), "Table A2. Allottee post-money stakes")
print("NOTE (manuscript Table A1): the printed upstream counts 393 (371 equity + 22 CB) and '353 structured + 7")
print("document-parsed' come from the v2 extraction; the canonical v3 universe is 415 (389 + 26) with 382 dated")
print("(357 structured + 25 parsed), identical from the 360-in-window step onward (321/260/210/209/208/201/196).")
print("This discrepancy is flagged in 10_submission/EDIT_SUGGESTIONS_2026-09-02.md for correction before submission.")


Sample flow (canonical universe v3)
-----------------------------------
                                                     step   n                                                                              note
     Treatment set (first third-party allotment per firm) 415                                      389 paid-in increases · 26 convertible bonds
                             With identifiable event date 382                                                  r1b 357 · document-parsed r1c 25
                       Within the stated 2015–2025 window 360 22 dated events fall outside it (18 before 2015, 4 in 2026) and enter no analysis
          Event month inside NPS panel (2015-11..2026-05) 321                                                       61 precede/follow the panel
               Window feasible (e−13 ≥ start, e+12 ≤ end) 260                                                                                  
                          Employment sample, primary rule 210   

## B1-B2 — Counterfactual audit and pseudo-event grid


In [ ]:
e = A("wp11e")
m = e["G4"]["oos_mse"]
rows = [{"Counterfactual model": lab, "Out-of-sample MSE": f"{m[k]:.4f}"} for k, lab in
        (("M1_match", "Matched listed controls"), ("M3_industry", "Industry benchmark"),
         ("M4_synth", "Synthetic control"), ("M2_trend", "Firm-specific linear trend"))]
show(pd.DataFrame(rows), "Table B1. Pre-event counterfactual-model comparison")
okp(m["M1_match"], 0.0393); okp(m["M3_industry"], 0.0490, 4)
print(f"B.2 prose: event-window {f4(e['G4']['event_effect'])} {ci4(e['G4']['ci'])} (n={e['G4']['n_event']}) · "
      f"trajectory-break {f4(e['G5a_trajbreak']['tau_accel'])} {ci4(e['G5a_trajbreak']['ci'])} · "
      f"DR-DiD {f4(e['G5b_dr']['ATT_dr'])} {ci4(e['G5b_dr']['ci'])}")
fg = A("wp11fg"); g = fg["g_placebo_grid"]; fh = fg["f_honest"]
rows = [{"Event clock": c, "N": g[c]["n"], "Mean gap": f4(g[c]["mean"]), "Mean CI": ci4(g[c]["mean_ci"]),
         "Tail excess": f4(g[c]["tail"]), "Tail CI": ci4(g[c]["tail_ci"])} for c in ("t-36", "t-30", "t-24", "t-18")]
rows.append({"Event clock": "Actual event", "N": 210, "Mean gap": f4(fh["mean"]["effect"]), "Mean CI": ci4(fh["mean"]["grid"][0]["ci"]),
             "Tail excess": f4(fh["tail"]["effect"]), "Tail CI": ci4(fh["tail"]["grid"][0]["ci"])})
show(pd.DataFrame(rows), "Table B2. Pseudo-event grid")


Table B1. Pre-event counterfactual-model comparison
---------------------------------------------------
      Counterfactual model Out-of-sample MSE
   Matched listed controls            0.0393
        Industry benchmark            0.0490
         Synthetic control            0.1067
Firm-specific linear trend            0.1368

B.2 prose: event-window -0.0954 [-0.1738, -0.0212] (n=136) · trajectory-break -0.0787 [-0.1342, -0.0265] · DR-DiD -0.0711 [-0.1223, -0.0222]
Table B2. Pseudo-event grid
---------------------------
 Event clock   N Mean gap            Mean CI Tail excess            Tail CI
        t-36 125  -0.0058 [-0.0435, +0.0307]     -0.0013 [-0.0250, +0.0283]
        t-30 131  -0.0115 [-0.0423, +0.0225]     -0.0142 [-0.0247, +0.0031]
        t-24 142  -0.0520 [-0.0927, -0.0157]     -0.0050 [-0.0211, +0.0173]
        t-18 163  -0.0576 [-0.1004, -0.0178]     +0.0092 [-0.0152, +0.0375]
Actual event 210  -0.0809 [-0.1006, -0.0611]     +0.1099 [+0.0907, +0.1291]


## B4-B6 — Per-date contrasts, threshold grid (t-24), filing screens


In [ ]:
r = A("wp13c_pooled_placebo")["runs"]
rows = [{"Comparison": f"Event minus t-{s}", "P10 difference": f4(r[f"D_t{s}_only"]["p10"]["obs"]),
         "95% CI": ci4(r[f"D_t{s}_only"]["p10"]["ci"]), "N pseudo": r[f"D_t{s}_only"]["n_placebo"]}
        for s in (18, 24, 30, 36)]
show(pd.DataFrame(rows), "Table B4. Tenth-percentile contrast by pseudo-date")
okp(r["D_t18_only"]["p10"]["obs"], -0.2035); okp(r["D_t24_only"]["p10"]["obs"], -0.2647)
d = A("wp11d"); G = d["grid"]
rows = []
for c in (-0.60, -0.35, -0.10):
    j = G.index(round(c, 2))
    rows.append({"Threshold": c, "Difference": f4(d["ddd"]["point"][j]),
                 "95% uniform band": f"[{d['ddd']['lo_unif'][j]:+.4f}, {d['ddd']['hi_unif'][j]:+.4f}]"})
show(pd.DataFrame(rows), "Table B5. Collapse-probability curve (event minus t-24; manuscript label 't-36' is corrected in the edits)")
jm = d["joint_multiplicity"]
print("max-t adjusted p:", dict(zip(jm["stats"], jm["maxT_adjusted_p"])))
okp(d["ddd"]["point"][G.index(-0.35)], 0.1199); assert jm["maxT_adjusted_p"][jm["stats"].index("p10")] == 0.0055
b6 = A("wp11o_confound")["employment"]
rows = [{"Sample": lab, "N": v["n"], "Mean": f4(v["mean"]), "95% CI": ci4(v["mean_ci"]), "Median": f4(v["median"]),
         "P10": f4(v["p10"]), "Pr<= -0.35": f"{v['collapse_035']:.4f}"}
        for lab, v in (("Full employment sample", b6["all"]), ("Narrow exclusion screen", b6["clean_narrow"]),
                       ("Broad exclusion screen", b6["clean_broad"]))]
show(pd.DataFrame(rows), "Table B6. Concurrent-filing screens")


Table B4. Tenth-percentile contrast by pseudo-date
--------------------------------------------------
      Comparison P10 difference             95% CI  N pseudo
Event minus t-18        -0.2035 [-0.3286, -0.0047]       163
Event minus t-24        -0.2647 [-0.3980, -0.0913]       142
Event minus t-30        -0.2969 [-0.4279, -0.1234]       131
Event minus t-36        -0.3060 [-0.4249, -0.1018]       125

Table B5. Collapse-probability curve (event minus t-24; manuscript label 't-36' is corrected in the edits)
----------------------------------------------------------------------------------------------------------
 Threshold Difference   95% uniform band
     -0.60    +0.0459 [+0.0006, +0.0912]
     -0.35    +0.1199 [+0.0515, +0.1884]
     -0.10    +0.1523 [+0.0417, +0.2630]

max-t adjusted p: {'mean': 0.7, 'p10': 0.0055, 'p25': 0.2975, 'median': 1.0}
Table B6. Concurrent-filing screens
-----------------------------------
                 Sample   N    Mean             95% CI  Median  

## C — Same-state balance, weights, alternative stratification (C1, C2, C5)


In [ ]:
c = A("wp12c_balance")["runs"]
d1, d0 = c["B_distress_1"], c["B_distress_0"]
lab = {"logsize": "Log firm size", "pg": "Pre-event employment growth", "yr": "Calendar year", "lev": "Leverage",
       "roa": "ROA", "cash": "Cash", "imp": "Capital impairment", "loss": "Loss"}
rows = [{"Covariate": lab[k], "Distressed": f"{d1['smd_by_covariate'][k]:+.4f}",
         "Non-distressed": (f"{d0['smd_by_covariate'][k]:+.4f}" if k in d0["smd_by_covariate"] else "-")}
        for k in lab]
show(pd.DataFrame(rows), "Table C1. Standardized mean differences after same-state weighting")
okp(d1["smd_by_covariate"]["roa"], -0.123, 3); okp(d1["max_abs_smd"], 0.123, 3)
wd1, wd0 = d1["weight_diagnostics"], d0["weight_diagnostics"]
rows = [{"Diagnostic": k, "Distressed": f"{wd1.get(k, d1.get(k)):,}", "Non-distressed": f"{wd0.get(k, d0.get(k)):,}"}
        for k in sorted(set(list(wd1) + ["ess"]))][:8]
show(pd.DataFrame(rows), "Table C2. Weight diagnostics (raw fields)")
assert round(d1["ess"], 1) == 14566.5
b = A("wp12b")["runs"]
rows = [{"State": lab, "Recipients": r_["n_treated"], "Median diff": f"{f4(r_['median']['obs'])} {ci4(r_['median']['ci'])}",
         "P10 diff": f"{f4(r_['p10']['obs'])} {ci4(r_['p10']['ci'])}"}
        for lab, r_ in (("Earlier employment decline", b["C_declfar_1"]), ("No earlier decline", b["C_declfar_0"]),
                        ("Pooled across groups", b["E_pool_declfar"]))]
show(pd.DataFrame(rows), "Table C5. Alternative stratification by earlier employment decline")
okp(b["C_declfar_1"]["p10"]["obs"], -0.1823); okp(b["C_declfar_0"]["p10"]["obs"], -0.2886)


Table C1. Standardized mean differences after same-state weighting
------------------------------------------------------------------
                  Covariate Distressed Non-distressed
              Log firm size    -0.0360        -0.0050
Pre-event employment growth    +0.0210        +0.0020
              Calendar year    -0.0010        -0.0020
                   Leverage    +0.0100        -0.0000
                        ROA    -0.1230        +0.0040
                       Cash    +0.0330        +0.0090
         Capital impairment    -0.0040              -
                       Loss    +0.0090              -

Table C2. Weight diagnostics (raw fields)
-----------------------------------------
   Diagnostic Distressed Non-distressed
          ess   14,566.5       68,284.7
    ess_share     0.5312         0.8657
max_over_mean        7.1            2.3
       n_ctrl     27,423         78,874
  top10_share     0.0026         0.0003
   top1_share    0.00026          3e-05
top1pct_share  

## D1, E1 — Observation window and flow decomposition


In [ ]:
w = A("wp13b_censoring")
rows = [{"Sample": lab, "N": v["n"], "Mean": f4(v["mean"]), "Median": f4(v["median"]), "P10": f4(v["p10"]),
         "Pr<=-0.35": f"{v['c35']:.4f}", "Pr<=-0.60": f"{v['c60']:.4f}"}
        for lab, v in (("Primary sample", w["full"]), ("Complete 12-month follow-up", w["complete12"]),
                       ("Nine early cessations at -0.75", w["worst_case_exits_as_collapse"]))]
show(pd.DataFrame(rows), "Table D1. Observation-window sensitivity")
okp(w["full"]["c35"], 0.1429); okp(w["worst_case_exits_as_collapse"]["c35"], 0.1781)
h = A("wp13h_flow_bands")["runs"]
f = h["E_flow_all"]
rows = [{"Statistic": lab, "Estimate": f4(f[k]["obs"]), "95% CI": ci4(f[k]["ci"])}
        for k, lab in (("hire", "Cumulative hires / baseline employment"),
                       ("sep", "Cumulative separations / baseline employment"),
                       ("diff", "Hires minus separations"))]
show(pd.DataFrame(rows), "Table E1. Employment-flow decomposition")
okp(f["hire"]["obs"], -0.0445); okp(f["diff"]["obs"], -0.0681); assert f["n"] == 217
ident = h["E_identity"]
print(f"E.3 reconciliation: n={ident['n']}, corr {ident['corr']:.4f}, median |gap| {ident['median_abs_gap']:.4f}, mean |gap| {ident['mean_abs_gap']:.4f}")
print("all appendix checks passed")


Table D1. Observation-window sensitivity
----------------------------------------
                        Sample   N    Mean  Median     P10 Pr<=-0.35 Pr<=-0.60
                Primary sample 210 -0.0304 +0.0292 -0.4360    0.1429    0.0571
   Complete 12-month follow-up 208 -0.0265 +0.0328 -0.4247    0.1394    0.0529
Nine early cessations at -0.75 219 -0.0599 +0.0149 -0.5601    0.1781    0.0959

Table E1. Employment-flow decomposition
---------------------------------------
                                   Statistic Estimate             95% CI
      Cumulative hires / baseline employment  -0.0445 [-0.1265, +0.0353]
Cumulative separations / baseline employment  +0.0236 [-0.0269, +0.0757]
                     Hires minus separations  -0.0681 [-0.1282, -0.0153]

E.3 reconciliation: n=208, corr 0.9245, median |gap| 0.0376, mean |gap| 0.0687
all appendix checks passed
